## Using Gemini to Interpret Machine Learning model

### Overview

This notebook creates a proof of concept approach for using a large language model to interpret the results of a logistic regression model that uses credit card data to predict if a purchase is fraudulent.  The purpose was largely to gain experience with the LLM element and add that skill to my toolset, so the logistic regression model isn't tuned to provide optimal performance.

The business use for a similar workflow would be to enable less technical users to gain a deeper understanding of how models in use within their company are reaching their predictions in situations where the data scientists aren't available directly.

Data files can be downloaded from https://www.kaggle.com/datasets/kaushalnandania/credit-card-fraud-detection/data and placed in the 'data' folder to run the notebook. A Gemini API key will also need to be placed in the .env file, a free-tier key will suffice.

In [107]:
import warnings
import pandas as pd
from google import genai
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

In [57]:
# load train and test data
train_df = pd.read_csv('data/train.csv', index_col=False)
test_df = pd.read_csv('data/test.csv', index_col=False)

# drop index column
train_df = train_df.drop(columns=['Unnamed: 0'])
test_df = test_df.drop(columns=['Unnamed: 0'])

### Data Exploration

In [40]:
# view the first few rows of the training data
display(train_df.head())

# print the shape of the training and test data
print(f'Training data shape: {train_df.shape}')
print(f'Test data shape: {test_df.shape}')

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


Training data shape: (1296675, 22)
Test data shape: (555719, 22)


In [109]:
# display all column names
pd.DataFrame({'Column Name': train_df.columns, 'Data Type': train_df.dtypes.values})

,Column Name,Data Type
0,trans_date_trans_time,object
1,cc_num,int64
2,merchant,object
3,category,object
4,amt,float64
5,first,object
6,last,object
7,gender,object
8,street,object
9,city,object


Several of the included columns won't provide value to our logistic regression model, so they need to be filtered out. Additionally, calculations/extractions need to be performed on numerical columns.

In [50]:
# create a copy of the training data to work with
train_data = train_df.copy()

# drop columns that won't be used for modeling
train_data = train_data.drop(columns=['cc_num', 'merchant', 'first', 'last', 'street', 'city', 'state', 'zip', 'job', 'trans_num'])

# extract year from dob and drop dob column
train_data['dob_year'] = pd.to_datetime(train_data['dob'], errors='coerce').dt.year
train_data = train_data.drop(columns=['dob'])

# convert unix_time to datetime and extract year
train_data['transaction_year'] = pd.to_datetime(train_data['unix_time'], unit='s', errors='coerce').dt.year

# calculate the age of the cardholder at the time of the transaction and drop dob_year and transaction_year columns
train_data['age'] = train_data['transaction_year'] - train_data['dob_year']
train_data = train_data.drop(columns=['dob_year', 'transaction_year', 'unix_time'])

# calculate distance from home to merchant using lat and long and drop lat and long columns
train_data['distance_from_home'] = ((train_data['merch_lat'] - train_data['lat']) ** 2 + (train_data['merch_long'] - train_data['long']) ** 2) ** 0.5
train_data = train_data.drop(columns=['lat', 'long', 'merch_lat', 'merch_long'])

# extract year, month, day, hour from transaction_date and drop transaction_date column
train_data['trans_year'] = pd.to_datetime(train_data['trans_date_trans_time'], errors='coerce').dt.year
train_data['trans_month'] = pd.to_datetime(train_data['trans_date_trans_time'], errors='coerce').dt.month
train_data['trans_day'] = pd.to_datetime(train_data['trans_date_trans_time'], errors='coerce').dt.day
train_data['trans_hour'] = pd.to_datetime(train_data['trans_date_trans_time'], errors='coerce').dt.hour
train_data = train_data.drop(columns=['trans_date_trans_time'])

# convert gender to binary
train_data['gender'] = train_data['gender'].map({'M': 0, 'F': 1})

# one-hot encode the category column
train_data = pd.get_dummies(train_data, columns=['category'])

Repeat the same process for the testing dataset. Duplicating so much code like this isn't ideal in a production environment, but for POC purposes it will suffice here.

In [51]:
# create a copy of the testing data to work with
test_data = test_df.copy()

# drop columns that won't be used for modeling
test_data = test_data.drop(columns=['cc_num', 'merchant', 'first', 'last', 'street', 'city', 'state', 'zip', 'job', 'trans_num'])

# extract year from dob and drop dob column
test_data['dob_year'] = pd.to_datetime(test_data['dob'], errors='coerce').dt.year
test_data = test_data.drop(columns=['dob'])

# convert unix_time to datetime and extract year
test_data['transaction_year'] = pd.to_datetime(test_data['unix_time'], unit='s', errors='coerce').dt.year

# calculate the age of the cardholder at the time of the transaction and drop dob_year and transaction_year columns
test_data['age'] = test_data['transaction_year'] - test_data['dob_year']
test_data = test_data.drop(columns=['dob_year', 'transaction_year', 'unix_time'])

# calculate distance from home to merchant using lat and long and drop lat and long columns
test_data['distance_from_home'] = ((test_data['merch_lat'] - test_data['lat']) ** 2 + (test_data['merch_long'] - test_data['long']) ** 2) ** 0.5
test_data = test_data.drop(columns=['lat', 'long', 'merch_lat', 'merch_long'])

# extract year, month, day, hour from transaction_date and drop transaction_date column
test_data['trans_year'] = pd.to_datetime(test_data['trans_date_trans_time'], errors='coerce').dt.year
test_data['trans_month'] = pd.to_datetime(test_data['trans_date_trans_time'], errors='coerce').dt.month
test_data['trans_day'] = pd.to_datetime(test_data['trans_date_trans_time'], errors='coerce').dt.day
test_data['trans_hour'] = pd.to_datetime(test_data['trans_date_trans_time'], errors='coerce').dt.hour
test_data = test_data.drop(columns=['trans_date_trans_time'])

# convert gender to binary
test_data['gender'] = test_data['gender'].map({'M': 0, 'F': 1})

# one-hot encode the category column
test_data = pd.get_dummies(test_data, columns=['category'])

In [52]:
# view the first few rows of the processed training data
display(train_data.head())

,amt,gender,city_pop,is_fraud,age,distance_from_home,trans_year,trans_month,trans_day,trans_hour,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,4.97,1,3495,0,24,0.872830,2019,1,1,0,...,False,False,False,False,True,False,False,False,False,False
1,107.23,1,149,0,34,0.272310,2019,1,1,0,...,True,False,False,False,False,False,False,False,False,False
2,220.11,0,4154,0,50,0.975845,2019,1,1,0,...,False,False,False,False,False,False,False,False,False,False
3,45.00,0,1939,0,45,0.919802,2019,1,1,0,...,False,False,False,False,False,False,False,False,False,False
4,41.96,0,99,0,26,0.868505,2019,1,1,0,...,False,False,False,False,False,True,False,False,False,False


### Train and evaluate model

In [ ]:
# train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(train_data.drop(columns=['is_fraud']), train_data['is_fraud'])

In [ ]:
# evaluate the model on the test data
test_predictions = model.predict(test_data.drop(columns=['is_fraud']))
accuracy = accuracy_score(test_data['is_fraud'], test_predictions)
print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.9956


This accuracy is concerningly high, but fine-tuning the logistic regression model or experimenting with potential alternative approaches isn't in the scope of this notebook.

### Use Gemini to interpret model

In [86]:
# randomly select 2 samples from each is_fraud class
sample_data = test_data.groupby('is_fraud', group_keys=False).apply(lambda x: x.sample(n=2, random_state=12))
display(sample_data)

,amt,gender,city_pop,is_fraud,age,distance_from_home,trans_year,trans_month,trans_day,trans_hour,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
50608,57.98,0,198,0,54,0.870855,2020,7,8,9,...,False,False,False,False,False,False,False,False,False,False
467078,6.22,0,301,0,46,0.571338,2020,12,12,18,...,False,False,False,False,False,False,True,False,False,False
93776,943.97,1,1606,1,47,1.149486,2020,7,23,23,...,False,False,False,False,False,False,False,True,False,False
194922,21.06,1,1970,1,28,0.831006,2020,8,28,23,...,False,False,False,False,False,False,True,False,False,False


In [87]:
# get feature names
feature_names = train_data.drop(columns=['is_fraud']).columns

# get coefficients and intercept
coefficients = model.coef_[0]
intercept = model.intercept_[0]

# print the logistic regression equation
equation = f"logit(p) = {intercept:.4f} "
for name, coef in zip(feature_names, coefficients):
    equation += f"+ ({coef:.4f} * {name}) "
print(equation)

logit(p) = 0.0001 + (0.0024 * amt) + (-0.6406 * gender) + (0.0000 * city_pop) + (0.0172 * age) + (0.0371 * distance_from_home) + (-0.0036 * trans_year) + (-0.0497 * trans_month) + (0.0058 * trans_day) + (0.0955 * trans_hour) + (-0.4214 * category_entertainment) + (-0.5117 * category_food_dining) + (0.1482 * category_gas_transport) + (-0.0423 * category_grocery_net) + (1.9564 * category_grocery_pos) + (-0.5357 * category_health_fitness) + (-0.7725 * category_home) + (-0.6218 * category_kids_pets) + (0.9527 * category_misc_net) + (-0.2626 * category_misc_pos) + (-0.4300 * category_personal_care) + (1.4619 * category_shopping_net) + (-0.1857 * category_shopping_pos) + (-0.7355 * category_travel) 


In [102]:
# use model to classify the three random samples from the test set
sample_predictions = model.predict(sample_data.drop(columns=['is_fraud']))
print(f'Sample Predictions: {sample_predictions}')

Sample Predictions: [0 0 0 0]


I had intentionally chosen to select two data points from each is_fraud group to have Gemini interpret, so I definitely don't like that our logistic regression model classified all four of them as 0 (not fraud). Combined with the extremely high accuracy, I'm wondering if there was a large class imbalance creating a situation where the model can nealy always predict 0.

In [106]:
# read api key from environment/.env file
with open('.env', encoding='utf-8') as f:
    for line in f:
        if line.strip().startswith('API_KEY='):
            api_key = line.strip().split('=', 1)[1].strip().strip('"').strip("'")
            break 

client = genai.Client(api_key=api_key)

We want to be as specific as possible with our prompt so Gemini doesn't have to make any assumptions that would cause inconsistent results.

In [110]:
# define prompt
prompt = f'''
The following is a logistic regression model that predicts the probability of a transaction being fraudulent based on various 
features of the transaction and the cardholder. The equation of the model is:
{equation}.
The model was trained on a dataset of credit card transactions, and it achieved an accuracy of {accuracy:.4f} on the test set.
The following are samples from the test set, along with their true labels and the model's predictions:
{sample_data[['is_fraud']].assign(prediction=sample_predictions)}.
Act as the data scientist that created the model and explain the model's predictions for these samples to a non-technical audience 
that doesn't need to know the mathematical details, and identify which features were most influential in the model's decision-making 
process for each sample.
'''

# call Gemini
response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=prompt
)
print(response.text)

Hello everyone. As the data scientist behind this fraud detection system, I want to walk you through how our model thinks and why it made specific decisions on these four transactions.

### How the Model "Thinks"
Think of this model as a digital security guard with a checklist. For every transaction, the guard looks at details like the price, how far you are from home, what you're buying, and even the time of day. 

Some things act as **"Red Flags"** that increase the suspicion score:
*   **Online Shopping:** Categories like "shopping_net" and "grocery_pos" (point-of-sale) carry a lot of weight in flagging fraud.
*   **Distance:** The further you are from home, the more suspicious the model becomes.
*   **Time:** Transactions late at night or early in the morning (high `trans_hour`) get more "points" toward a fraud prediction.

Some things act as **"Safety Signs"** that lower the suspicion score:
*   **Categories:** Buying things for the home, travel, or dining out are patterns the mod

This result came after several iterations of improving the prompt to add more detail in a few spots. While it isn't exactly perfect, I think it showcases the capability that I had sought out to test. Interestingly, minor changes in the prompt resulted in significant changes in the phrasing of the response which really highlights the probabilistic nature of the LLM and serves as a reminder that we'd want to include some guardrails and/or reminders for users that results may not always be entirely accurate.

Real-world implementations that I could see being valuable could be providing a high level summary what features a model uses and how their values effect the output on executive or operational dashboards or in a situation where there's individual tickets that a human would typically respond to (support, security incidents, etc.) a data scientist could create a model to predict the outcome and have the LLM explain to the human in the loop how it got to a classification.